In [1]:
# 🍬 WishWhisk V1: 我的心愿雷达配置

# wishlist 是一个列表 [ ]，里面装着我们所有的任务卡片 { }
wishlist = [
    {
        "name": "Lemaire scarf bag", 
        "keywords": ["lemaire", "scarf", "black"], 
        "target_price": 1350  
    },
    {
        "name": "Lemaire large croissant bag dark chocolate",
        "keywords": ["lemaire", "croissant", "large", "brown"], 
        "target_price": 1400 
    },
    
]

print(f"✅ Successfully loaded {len(wishlist)} tracking tasks! Ready to hunt.")

✅ Successfully loaded 2 tracking tasks! Ready to hunt.


In [2]:
import os

def notify_mac(title, text):
    # Use os.system to trigger macOS's built-in AppleScript for notifications
    os.system(f"""osascript -e 'display notification "{text}" with title "{title}"'""")

# Test it out right away!
notify_mac("TreatTracker 🍬", "Deal alert! Your Lemaire Croissant dropped below $1000. Go get it!")

In [3]:
def build_search_url(keywords):
    # Join the keywords using a plus sign "+"
    search_query = "+".join(keywords)
    
    # Combine the base URL with our query
    full_url = f"https://www.ssense.com/en-ca/women?q={search_query}"
    
    return full_url

# Let's test if our machine works!
test_keywords = ["lemaire", "croissant", "small", "black"]
generated_url = build_search_url(test_keywords)

print("Generated URL:")
print(generated_url)

Generated URL:
https://www.ssense.com/en-ca/women?q=lemaire+croissant+small+black


In [4]:
# We import the 'requests' from our newly installed elite toolkit
from curl_cffi import requests

def fetch_webpage(url):
    print(f"Sending the elite agent to: {url} ...")
    
    # The magic happens here: impersonate="chrome110" 
    # This perfectly mimics the exact walking posture of a real Chrome browser!
    response = requests.get(url, impersonate="chrome110")
    
    if response.status_code == 200:
        print("✅ Success! The security guard was fooled. We got the data.")
        print("-" * 30)
        print("Here is a peek at the HTML blueprint:")
        print(response.text[:300]) 
        print("-" * 30)
        return response.text
    elif response.status_code == 403:
        print("🛑 Oh no! The guard STILL kicked us out (403 Forbidden).")
        return None
    else:
        print(f"⚠️ Something weird happened. Status Code: {response.status_code}")
        return None

# Let's test it again with our URL!
test_keywords = ["lemaire", "croissant", "small", "black"]
test_url = build_search_url(test_keywords)
html_data = fetch_webpage(test_url)

Sending the elite agent to: https://www.ssense.com/en-ca/women?q=lemaire+croissant+small+black ...
✅ Success! The security guard was fooled. We got the data.
------------------------------
Here is a peek at the HTML blueprint:
<!DOCTYPE html><html lang="en-ca"><head><meta name="language" content="en"><meta http-equiv="Content-Type" content="text/html; charset=utf-8"><title>Designer Clothes, Shoes & Bags for Women | SSENSE Canada</title><meta name="viewport" content="width=device-width,initial-scale=1,maximum-scale=1,user-
------------------------------


In [5]:
from bs4 import BeautifulSoup

def extract_items_from_list(html_data):
    print("🥣 Giving the blueprint to BeautifulSoup...")
    soup = BeautifulSoup(html_data, 'html.parser')
    
    # 1. THE MAGIC FIX: Use 'select' with '^=' which means "starts with"
    # Find all spans where data-test starts with "productName"
    name_boxes = soup.select('span[data-test^="productName"]')
    price_boxes = soup.select('span[data-test^="productCurrentPrice"]')
    
    if not name_boxes or not price_boxes:
        print("⚠️ Couldn't find items on the shelf.")
        return []
        
    print(f"🎯 Jackpot! Found {len(name_boxes)} items on this page.")
    
    extracted_items = []
    
    # 2. Pair them up and clean the price just like before
    for name_box, price_box in zip(name_boxes, price_boxes):
        raw_name = name_box.text.strip()
        raw_price = price_box.text.strip()
        
        clean_text = raw_price.replace('$', '').replace(',', '').replace('CAD', '').strip()
        
        if clean_text:
            clean_price = float(clean_text)
            extracted_items.append({
                "name": raw_name,
                "price": clean_price
            })
            print(f"🛍️ Found: {raw_name} : ${clean_price}")
            
    return extracted_items

# Let's test the fixed extractor!
if html_data:
    all_items = extract_items_from_list(html_data)

🥣 Giving the blueprint to BeautifulSoup...
🎯 Jackpot! Found 2 items on this page.
🛍️ Found: Black Small Croissant Bag : $785.0
🛍️ Found: Black Small Croissant Bag : $1220.0


In [6]:
import time

def run_treat_tracker():
    print("🚀 WishWhisk is starting its patrol...\n")
    
    # Loop through each item in your wishlist
    for item in wishlist:
        name = item["name"]
        target_price = item["target_price"]
        
        # 1. Build the exact search URL
        search_query = "+".join(item["keywords"])
        url = f"https://www.ssense.com/en-ca/women?q={search_query}"
        
        print(f"🔍 Checking: {name} (Target: ${target_price})")
        
        # 2. Fetch the blueprint
        html_data = fetch_webpage(url)
        
        if not html_data:
            print("  ⏭️ Skipping to the next item due to an error.")
            continue
            
        # 3. Extract all bags from the shelf
        all_bags = extract_items_from_list(html_data)
        
        # 4. Check for deals!
        deal_found = False
        for bag in all_bags:
            bag_name = bag["name"]
            bag_price = bag["price"]
            
            # If the price is lower than or equal to our target...
            if bag_price <= target_price:
                print(f"  🎉 DEAL ALERT! '{bag_name}' is only ${bag_price}!")
                # Sound the alarm!
                notify_mac("TreatTracker 🍬", f"Deal alert! {bag_name} dropped to ${bag_price}!")
                deal_found = True
                break # We found a deal, no need to check other bags on this page
                
        if not deal_found:
            print(f"  🥲 No deals under ${target_price} yet.")
            
        print("-" * 40)
        
        # 5. Be polite: wait 3 seconds before checking the next item
        time.sleep(3)

    print("🏁 Patrol finished! See you next time.")

# 💥 The moment of truth: Run the whole system!
run_treat_tracker()

🚀 WishWhisk is starting its patrol...

🔍 Checking: Lemaire scarf bag (Target: $1350)
Sending the elite agent to: https://www.ssense.com/en-ca/women?q=lemaire+scarf+black ...
✅ Success! The security guard was fooled. We got the data.
------------------------------
Here is a peek at the HTML blueprint:
<!DOCTYPE html><html lang="en-ca"><head><meta name="language" content="en"><meta http-equiv="Content-Type" content="text/html; charset=utf-8"><title>Designer Clothes, Shoes & Bags for Women | SSENSE Canada</title><meta name="viewport" content="width=device-width,initial-scale=1,maximum-scale=1,user-
------------------------------
🥣 Giving the blueprint to BeautifulSoup...
🎯 Jackpot! Found 3 items on this page.
🛍️ Found: Black Foulard T-shirt : $310.0
🛍️ Found: Black & Brown Small Scarf Extended Handle Bag : $1690.0
🛍️ Found: Black Small Scarf Extended Handle Bag : $1130.0
  🎉 DEAL ALERT! 'Black Foulard T-shirt' is only $310.0!
----------------------------------------
🔍 Checking: Lemaire la